In [2]:
import numpy as np
import random
import time
import os

# Attempt to load PIL for GIF generation
try:
    from PIL import Image, ImageDraw, ImageFont
    PIL_AVAILABLE = True
except ImportError:
    PIL_AVAILABLE = False
    print("Pillow not found. GIF generation will be skipped. (pip install Pillow numpy)")

# ==========================================
# TERMINAL UI COLORS
# ==========================================
class Colors:
    RED = '\033[91m'     
    YELLOW = '\033[93m'  
    WHITE = '\033[97m'   
    GREEN = '\033[92m'   
    CYAN = '\033[96m'    
    RESET = '\033[0m'

# ==========================================
# ENVIRONMENT CLASS
# ==========================================
class F1TeamEnv:
    def __init__(self, total_laps=60):
        self.driver_map = {
            "BlueCow":     [("VER", "Max Versplatton"), ("LAW", "Liam Awesome")],
            "Merciless":   [("RUS", "Forge Hustle"),    ("ANT", "Kimi Macaroni")],
            "Furrari":     [("LEC", "Chuck LeClutch"),  ("HAM", "Louis Hamstring")],
            "McPapaya":    [("NOR", "Lando Chuckris"),  ("PIA", "Osco Pastry")],
            "Astonishing": [("ALO", "Nando Alfonso"),   ("STR", "Lance Scroll")],
            "Alpain":      [("GAS", "Peter Ghastly"),   ("COL", "Franky Colapunch")],
            "Billiams":    [("ALB", "Alex Album"),      ("SAI", "Carlos Signs")],
            "ToroLoco":    [("HAD", "Isaac Badger"),    ("LIN", "Artie Lindblad")],
            "Sober":       [("HUL", "Nico Bulkensmear"),("BOR", "Gabe Tortellini")],
            "Hassle":      [("OCO", "Esteban Acorn"),   ("BEA", "Ollie Birdman")],
            "CaddyShack":  [("PER", "Surge Perez"),     ("BOT", "Battery Voltas")]
        }
        self.teams = list(self.driver_map.keys())
        self.total_laps = total_laps
        self.reset() 

    def reset(self, seed=None):
        if seed is not None:
            random.seed(seed)
            np.random.seed(seed)
            
        self.current_lap = 0
        all_drivers = []
        for team_name in self.teams:
            for driver_idx in [0, 1]:
                drv_code, drv_name = self.driver_map[team_name][driver_idx]
                all_drivers.append({"id": drv_code, "full_name": drv_name, "team": team_name})
                
        random.shuffle(all_drivers)
        
        self.cars = []
        for grid_pos, driver_data in enumerate(all_drivers):
            driver_data.update({
                "total_race_time": grid_pos * 1.5, 
                "tyre_compound": 2,                
                "tyre_age": 0.0,
                "battery": 1.0,
                "pit_stops": 0,
                "last_lap_time": 0.0, 
                "status": "GRID",
                "current_pace_cmd": 1
            })
            self.cars.append(driver_data)

    def _process_car_lap(self, car, team_action):
        base_lap_time = 85.0 
        pit_loss_time = 25.0 
        lab_time_std = 0.01
        
        is_car_2 = (car["id"] == self.driver_map[car["team"]][1][0])
        pace_cmd = team_action[2] if is_car_2 else team_action[0]
        pit_cmd = team_action[3] if is_car_2 else team_action[1]
        
        car["status"] = "OUT"
        car["current_pace_cmd"] = pace_cmd 
        
        if pit_cmd > 0:
            car["total_race_time"] += pit_loss_time
            car["tyre_compound"] = pit_cmd 
            car["tyre_age"] = 0
            car["pit_stops"] += 1
            car["status"] = "PIT"
            
        pace_modifier = 0.0
        deg_modifier = 1.0
        
        if pace_cmd == 0:   
            pace_modifier = 1.5   
            car["battery"] = min(1.0, car["battery"] + 0.25)
            deg_modifier = 0.5
            if car["status"] != "PIT": car["status"] = "HRV"
            
        elif pace_cmd == 2: 
            if car["override_unlocked"] and car["battery"] > 0.2:
                pace_modifier = -1.2  
                car["battery"] -= 0.25 
                deg_modifier = 1.5    
                if car["status"] != "PIT": car["status"] = "OVR" 
            elif not car["override_unlocked"] and car["battery"] > 0.15:
                pace_modifier = -0.6
                car["battery"] -= 0.15 
                deg_modifier = 1.2
                if car["status"] != "PIT": car["status"] = "BST"
            else:
                pace_modifier = 0.0 
        else:
            if car["status"] != "PIT": car["status"] = "STD"
            
        tyre_pace_deltas = {1: -1.0, 2: 0.0, 3: 1.0}
        tyre_deg_rates = {1: 0.05, 2: 0.04, 3: 0.02}
        compound = car["tyre_compound"]
        
        deg_penalty = (car["tyre_age"] * tyre_deg_rates[compound]) ** 2 
        lap_noise = np.random.normal(0.0, lab_time_std)
        
        lap_time = base_lap_time + tyre_pace_deltas[compound] + deg_penalty + pace_modifier + lap_noise
        car["last_lap_time"] = lap_time
        car["total_race_time"] += lap_time
        car["tyre_age"] += (1.0 * deg_modifier)

    def _resolve_overtakes(self, grid_order):
        for i in range(1, len(grid_order)):
            attacker = grid_order[i]
            defender = grid_order[i-1]
            
            if attacker["total_race_time"] < defender["total_race_time"]:
                overtake_chance = 0.3 
                
                if attacker["status"] == "OVR": overtake_chance += 0.4 
                elif attacker["status"] == "BST": overtake_chance += 0.15
                if defender["status"] in ["OVR", "BST"]: overtake_chance -= 0.3 
                    
                tyre_delta = defender["tyre_age"] - attacker["tyre_age"]
                overtake_chance += (tyre_delta * 0.05)
                overtake_chance = max(0.05, min(0.95, overtake_chance))
                
                if random.random() < overtake_chance:
                    attacker["total_race_time"] += 0.5
                    defender["total_race_time"] += 0.5
                    attacker["last_lap_time"] += 0.5
                    defender["last_lap_time"] += 0.5
                else:
                    time_lost = (defender["total_race_time"] + 0.3) - attacker["total_race_time"]
                    attacker["total_race_time"] += time_lost
                    attacker["last_lap_time"] += time_lost

    def step(self, actions):
        self.current_lap += 1
        grid_order = list(self.cars) 
        
        for i, car in enumerate(grid_order):
            if i == 0:
                car["override_unlocked"] = False
            else:
                gap_to_car_ahead = car["total_race_time"] - grid_order[i-1]["total_race_time"]
                car["override_unlocked"] = (gap_to_car_ahead < 1.0)

        for car in self.cars:
            team_action = actions[car["team"]]
            self._process_car_lap(car, team_action)

        self._resolve_overtakes(grid_order)
        self.cars.sort(key=lambda x: x["total_race_time"])

    def format_time(self, seconds):
        if seconds <= 0: return "0:00.000"
        mins = int(seconds // 60)
        secs = seconds % 60
        return f"{mins}:{secs:06.3f}"

    def get_board_string(self):
        """Generates the telemetry board WITH ANSI color codes."""
        lines = []
        lines.append(f"{Colors.CYAN}========================================================================================{Colors.RESET}")
        # Updated to "Formula Mar1"
        lines.append(f"  FIA FORMULA MAR1 WORLD CHAMPIONSHIP - LAP {self.current_lap:2d} / {self.total_laps}  |  TRACK: GREEN  |  CARS: 22")
        lines.append(f"{Colors.CYAN}========================================================================================{Colors.RESET}")
        lines.append(" P  | DRIVER | GAP        | INT        | LAST LAP | TYRE | LAPS | BATTERY | STP | MODE")
        lines.append("----------------------------------------------------------------------------------------")
        
        leader_time = self.cars[0]["total_race_time"]
        
        for i, car in enumerate(self.cars):
            pos = f"{i+1:2d}"
            driver = f"{car['id']:<6}"
            
            if i == 0:
                gap, interval = "Leader    ", "-         "
            else:
                gap = f"+{car['total_race_time'] - leader_time:<9.3f}"
                interval = f"+{car['total_race_time'] - self.cars[i-1]['total_race_time']:<9.3f}"
                
            t_comp = car["tyre_compound"]
            tyre_str = f"{Colors.RED}S{Colors.RESET}" if t_comp == 1 else f"{Colors.YELLOW}M{Colors.RESET}" if t_comp == 2 else f"{Colors.WHITE}H{Colors.RESET}"
                
            ers_blocks = int((car["battery"] / 1.0) * 5)
            ers_bar = f"[{'|'*ers_blocks}{' '*(5-ers_blocks)}]"
            
            status = car['status']
            if status == "OVR":
                status = f"{Colors.GREEN}{status}{Colors.RESET}"
                
            row = f" {pos} | {driver} | {gap} | {interval} | {self.format_time(car['last_lap_time'])} |  {tyre_str}   |  {int(car['tyre_age']):2d}  | {ers_bar} |  {car['pit_stops']}  | {status}"
            lines.append(row)
            
        lines.append(f"{Colors.CYAN}========================================================================================{Colors.RESET}")
        # Added watermark/signature at the bottom
        lines.append(f"                                                            Simulation by Jan A. Krzywda")
        return "\n".join(lines)

    def render_telemetry(self):
        os.system('cls' if os.name == 'nt' else 'clear')
        print(self.get_board_string())


# ==========================================
# ADVANCED GIF GENERATOR HELPER
# ==========================================
def get_monospace_font(size=14):
    """Attempts to load a true monospace font to guarantee column alignment."""
    if not PIL_AVAILABLE: return None
    fallbacks = ["consola.ttf", "cour.ttf", "lucon.ttf", "Menlo.ttc", "Monaco.ttf", "DejaVuSansMono.ttf"]
    for font_name in fallbacks:
        try:
            return ImageFont.truetype(font_name, size)
        except IOError:
            continue
    return ImageFont.load_default()

def draw_ansi_text_to_image(text, font):
    """Parses ANSI colors and draws perfectly aligned colored text onto a PIL Image."""
    # Dark Mode Background - Height increased to 580 to fit the new signature line
    img = Image.new('RGB', (820, 580), color=(15, 15, 18)) 
    draw = ImageDraw.Draw(img)
    
    # Map ANSI codes to RGB tuples
    color_map = {
        '91m': (255, 85, 85),   # Red
        '93m': (255, 235, 85),  # Yellow
        '97m': (255, 255, 255), # White
        '92m': (85, 255, 85),   # Green
        '96m': (85, 255, 255),  # Cyan
        '0m':  (200, 200, 200)  # Default Gray
    }
    
    # Calculate exact character width to guarantee perfect column alignment
    char_width = font.getlength("A") if hasattr(font, 'getlength') else font.getbbox("A")[2] if hasattr(font, 'getbbox') else 8
    
    current_y = 15
    for line in text.split('\n'):
        current_x = 15
        current_color = color_map['0m']
        
        # Split line by ANSI escape sequence
        parts = line.split('\033[')
        for i, part in enumerate(parts):
            if i == 0:
                draw.text((current_x, current_y), part, fill=current_color, font=font)
                current_x += len(part) * char_width
            else:
                # Find where the color code ends (the 'm' character)
                code_end = part.find('m')
                if code_end != -1:
                    code = part[:code_end+1]
                    text_part = part[code_end+1:]
                    
                    if code in color_map:
                        current_color = color_map[code]
                        
                    draw.text((current_x, current_y), text_part, fill=current_color, font=font)
                    current_x += len(text_part) * char_width
                    
        current_y += 18 # Line spacing

    return img

# ==========================================
# MAIN EXECUTION LOOP
# ==========================================
if __name__ == "__main__":
    env = F1TeamEnv(total_laps=60) # Full 60 lap race
    frames = [] 
    
    if PIL_AVAILABLE:
        font = get_monospace_font(14)
    
    print("LIGHTS OUT AND AWAY WE GO! (Simulating 60 laps...)")
    time.sleep(1)
    
    while env.current_lap < env.total_laps:
        actions = {}
        for team in env.teams:
            pace1, pace2 = random.choice([0, 1, 2]), random.choice([0, 1, 2])
            
            # 94% Stay Out to prevent excessive pitting
            pit1 = np.random.choice([0, 1, 2, 3], p=[0.94, 0.02, 0.02, 0.02])
            pit2 = np.random.choice([0, 1, 2, 3], p=[0.94, 0.02, 0.02, 0.02])
            
            actions[team] = [pace1, pit1, pace2, pit2]
            
        env.step(actions)
        
        # 1. Output to Terminal
        env.render_telemetry()
        
        # 2. Render pretty frame for the GIF
        if PIL_AVAILABLE:
            board_str = env.get_board_string()
            frames.append(draw_ansi_text_to_image(board_str, font))
            
        time.sleep(0.3) 
        
    print("🏁 CHEQUERED FLAG! 🏁")
    
    # Compile and Save GIF
    if PIL_AVAILABLE and len(frames) > 0:
        print("\nCompiling high-quality GIF... (This may take a few seconds)")
        frames[0].save(
            'f1_2026_simulation.gif',
            save_all=True,
            append_images=frames[1:],
            optimize=True,
            duration=1000, 
            loop=0
        )
        print("f1_2026_simulation.gif saved successfully! You can upload this straight to WordPress.")

LIGHTS OUT AND AWAY WE GO! (Simulating 60 laps...)
  FIA FORMULA MAR1 WORLD CHAMPIONSHIP - LAP  1 / 60  |  TRACK: GREEN  |  CARS: 22
 P  | DRIVER | GAP        | INT        | LAST LAP | TYRE | LAPS | BATTERY | STP | MODE
----------------------------------------------------------------------------------------
  1 | PER    | Leader     | -          | 1:27.000 |  M   |   0  | [|||||] |  0  | HRV
  2 | LAW    | +0.300     | +0.300     | 1:25.800 |  M   |   1  | [|||||] |  0  | STD
  3 | SAI    | +0.999     | +0.699     | 1:25.000 |  M   |   1  | [|||||] |  0  | STD
  4 | ALB    | +1.899     | +0.899     | 1:24.399 |  M   |   1  | [|||| ] |  0  | BST
  5 | BEA    | +5.394     | +3.495     | 1:24.894 |  M   |   1  | [|||| ] |  0  | BST
  6 | VER    | +6.013     | +0.620     | 1:27.014 |  M   |   0  | [|||||] |  0  | HRV
  7 | HAM    | +8.391     | +2.377     | 1:24.891 |  M   |   1  | [|||| ] |  0  | BST
  8 | BOT    | +8.999     | +0.608     | 1:26.999 |  M   |   0  | [|||||] |  0  | HRV
  9